# Financial Market Statistical Analysis

This notebook provides a comprehensive statistical analysis of financial features using the refactored `finance_ml.analytics` module. 
It leverages advanced statistical methods (Bayesian, MCMC, Kalman Filters), interactive dashboards, and performance-optimized operations.


## 1. Setup and Environment Configuration
We import the core analytics modules and configure the visualization environment.


In [63]:
import logging
import warnings

import pandas as pd
import plotly.express as px

# Core analytics imports (consolidated)
from finance_ml.analytics import (
    # Data utilities
    backfill_feature_columns,
    # Feature categories (dynamic loading from DB)
    FEATURE_CATEGORIES,
    compare_registry_with_local,
    _get_fallback_feature_categories,
    # Statistical analysis
    bayesian_category_analysis,
    fit_distributions_by_category,
    hierarchical_mcmc_by_sector,
    kalman_momentum_filter,
    fit_gaussian_copula,
    # Optimized operations
    fast_ruin_probability,
    get_optimization_status,
    # Screening (including new screeners)
    create_enhanced_screener,
    screen_garp_opportunities,
    screen_high_yield_safe_dividends,
    screen_value_opportunities,
    screen_growth_momentum,
    # Feature analytics & dashboards
    PLOTLY_TEMPLATE,
    create_interactive_momentum_dashboard,
    create_interactive_valuation_heatmap,
    create_leverage_liquidity_quadrant,
    bayesian_earnings_beat_model,
    analyze_distress_distribution,
    create_summary_dashboard,
)
# Probability Analytics (Bayesian earnings beat, EPS streaks, model confidence)
from finance_ml.analytics.probability_analytics import (
    EarningsBeatProbabilityModel,
    EPSStreakAnalyzer,
    ModelConfidenceEstimator,
    create_earnings_probability_dashboard,
    create_confidence_calibration_chart,
    create_eps_streak_analysis_chart,
    export_probability_analytics_results,
)
# Category-specific chart functions
from finance_ml.analytics.visualizations.category_charts import (
    # Analyst Sentiment
    create_analyst_sentiment_histogram,
    create_analyst_upside_scatter,
    # Earnings Quality
    create_eps_surprise_histogram,
    create_eps_trajectory_scatter,
    # Growth Metrics
    create_growth_correlation_heatmap,
    create_revenue_vs_eps_growth_scatter,
    # Cash Flow
    create_fcf_margin_yield_scatter,
    create_cash_flow_quality_boxplot,
    # Dividend Features
    create_dividend_yield_payout_scatter,
    create_shareholder_yield_histogram,
    # R&D Investment
    create_rnd_intensity_boxplot,
    create_rnd_intensity_growth_scatter,
    create_rnd_per_employee_histogram,
    # Inventory
    create_inventory_days_turnover_scatter,
    # Goodwill & M&A
    create_goodwill_concentration_boxplot,
    create_goodwill_impairment_scatter,
    create_acquisition_activity_histogram,
    # CapEx & Investment
    create_capex_growth_scatter,
    create_investment_efficiency_boxplot,
    create_ma_intensity_histogram,
    # Advanced visualizations (new)
    create_valuation_violin_plot,
    create_quality_risk_radar_chart,
    create_leverage_liquidity_bubble_chart,
)
# Profitability visualizations
from finance_ml.analytics.visualizations.profitability import (
    create_margin_waterfall_chart,
    create_dupont_decomposition_dashboard,
    create_profitability_quadrant,
)
# Technical visualizations
from finance_ml.analytics.visualizations.technical import (
    create_momentum_ribbon_chart,
    create_52w_range_distribution,
)
# Temporal analysis visualizations
from finance_ml.analytics.visualizations.temporal_analysis import (
    create_earnings_calendar_heatmap,
    create_inventory_cycle_analysis,
    create_fcf_trajectory_chart,
    create_dividend_streak_timeline,
)

# Configuration
logging.basicConfig(level=logging.INFO)
warnings.filterwarnings("ignore")
px.defaults.template = PLOTLY_TEMPLATE

# Display optimization and feature category status
opt_status = get_optimization_status()
print(f"JIT Acceleration: {opt_status.get('numba_available')}")
print(f"Feature Categories Loaded: {len(FEATURE_CATEGORIES)}")


JIT Acceleration: False
Feature Categories Loaded: 14


## 2. Data Acquisition
Loading feature categories dynamically from the `calculated_features_registry` table (with fallback).
Data is loaded from the `mv_all_stock_features` materialized view.


In [64]:
# Feature categories are now loaded dynamically from the database at module import
# FEATURE_CATEGORIES is imported from finance_ml.analytics
# Compare with fallback to detect any drift
fallback_categories = _get_fallback_feature_categories()
diff_report = compare_registry_with_local(FEATURE_CATEGORIES, fallback_categories)

print(f"📊 Feature Categories Summary:")
print(f"   Categories loaded: {len(FEATURE_CATEGORIES)}")
print(f"   Total features: {sum(len(v) for v in FEATURE_CATEGORIES.values())}")

if diff_report["features_only_in_db"]:
    print("\n📌 New features in registry (not in fallback):")
    for cat, feats in diff_report["features_only_in_db"].items():
        print(f"   {cat}: {feats}")


📊 Feature Categories Summary:
   Categories loaded: 14
   Total features: 84


In [65]:
%%sql
SELECT *
FROM public.mv_all_stock_features
WHERE next_earnings >= DATE '2026-01-01' and region = 'Europe'
ORDER BY next_earnings ASC;


,isin,ticker,name,industry,sector,trading_country,region,country,exchange,last_updated,...,other_unusual_items_ltm,impairment_goodwill_ltm,unusual_asset_writedown_ltm,restructuring_charges_ltm,total_unusual_items,unusual_items_to_revenue,unusual_items_to_ebitda,has_unusual_items_flag,earnings_quality_impact,feature_calculated_at
0,FI0009007884,ELISA,Elisa Oyj,Diversified Telecommunication Services,Communication Services,FI,Europe,FI,HLSE,2026-01-30,...,0.00,0.00,0.00,0.0,0.00,0.000000,0.000000,0,100.000000,2026-02-01 17:19:21.268578+00
1,US0528001094,ALV,Autoliv Inc.,Automobile Components,Consumer Discretionary,US,Europe,SE,NYSE,2026-01-30,...,0.00,0.00,0.00,-20.0,-20.00,0.188430,1.328021,1,97.340426,2026-02-01 17:19:21.268578+00
2,SE0015949201,LIFCOB,Lifco AB (publ),Industrial Conglomerates,Industrials,SE,Europe,SE,OM,2026-01-30,...,0.00,0.00,0.00,0.0,0.00,0.000000,0.000000,0,100.000000,2026-02-01 17:19:21.268578+00
3,SE0000108227,SKFB,AB SKF (publ),Machinery,Industrials,SE,Europe,SE,OM,2026-01-30,...,-36.75,0.00,-36.75,0.0,-73.50,0.733501,5.785261,1,85.779240,2026-02-01 17:19:21.268578+00
4,BE0974256852,COLR,Colruyt Group N.V.,Consumer Staples Distribution and Retail,Consumer Staples,BE,Europe,BE,ENXTBR,2026-01-30,...,-2.93,-0.35,-2.93,0.0,-6.21,0.047258,0.735294,1,98.195869,2026-02-01 17:19:21.268578+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1896,CH0006372897,INRN,Interroll Holding AG,Machinery,Industrials,CH,Europe,CH,SWX,2026-01-30,...,0.00,0.00,0.00,0.0,0.00,0.000000,0.000000,0,100.000000,2026-02-01 17:19:21.268578+00
1897,CH0010702154,KOMN,Komax Holding AG,Machinery,Industrials,CH,Europe,CH,SWX,2026-01-30,...,0.40,0.00,0.00,0.0,0.40,0.054487,1.287416,1,96.927803,2026-02-01 17:19:21.268578+00
1898,DE0007276503,YSN,secunet Security Networks Aktiengesellschaft,IT Services,Information Technology,DE,Europe,DE,XTRA,2026-01-30,...,0.00,0.00,0.00,0.0,0.00,0.000000,0.000000,0,100.000000,2026-02-01 17:19:21.268578+00
1899,FR0004027068,ALLAN,Lanson-BCC,Beverages,Consumer Staples,FR,Europe,FR,ENXTPA,2026-01-30,...,0.00,0.00,0.00,0.0,0.00,0.000000,0.000000,0,100.000000,2026-02-01 17:19:21.268578+00


In [66]:
# Normalize SQL result and backfill expected columns using the utility function
if not isinstance(df, pd.DataFrame):
    try:
        df = df.DataFrame()
    except Exception:
        pass

if isinstance(df, pd.DataFrame):
    df = backfill_feature_columns(df)
    print(f"✓ Backfill complete. Columns: {len(df.columns)}")


✓ Backfill complete. Columns: 716


## 3. Comprehensive Statistical Analysis by Category

This section provides in-depth statistical analysis across all 14 feature categories using Bayesian methods, distribution fitting, and specialized visualizations.


### 3.1 Valuation Ratios
We use Bayesian analysis to estimate true valuation means and visualize valuation metrics across industries.


In [67]:
val_results = bayesian_category_analysis(df, 'Valuation Ratios', FEATURE_CATEGORIES['Valuation Ratios'])
val_distributions = fit_distributions_by_category(df, 'Valuation Ratios', FEATURE_CATEGORIES['Valuation Ratios'])
create_interactive_valuation_heatmap(df).show()


### 3.2 Momentum & Technical
Applying Kalman filters to smooth momentum signals and visualizing multi-period momentum patterns.


In [68]:
df_kalman = kalman_momentum_filter(df, momentum_cols=['price_momentum_1y', 'price_momentum_3m'])
momentum_results = bayesian_category_analysis(df, 'Momentum & Technical', FEATURE_CATEGORIES['Momentum & Technical'])
create_interactive_momentum_dashboard(df).show()

In [69]:
create_52w_range_distribution(df).show()

In [70]:
create_momentum_ribbon_chart(df).show()


### 3.3 Profitability
Utilizing DuPont decomposition and margin waterfall charts to analyze bottom-line drivers with Bayesian estimation.


In [71]:
prof_results = bayesian_category_analysis(df, 'Profitability', FEATURE_CATEGORIES['Profitability'])
prof_distributions = fit_distributions_by_category(df, 'Profitability', FEATURE_CATEGORIES['Profitability'])
create_dupont_decomposition_dashboard(df).show()

In [72]:
create_margin_waterfall_chart(df).show()

In [73]:
create_profitability_quadrant(df).show()


### 3.4 Quality & Risk
Assessing quality scores and distress risk using Bayesian analysis and tail risk metrics.


In [74]:
quality_results = bayesian_category_analysis(df, 'Quality & Risk', FEATURE_CATEGORIES['Quality & Risk'])
quality_distributions = fit_distributions_by_category(df, 'Quality & Risk', FEATURE_CATEGORIES['Quality & Risk'])
analyze_distress_distribution(df).show()
df_ruin = fast_ruin_probability(df)


### 3.5 Leverage & Liquidity
Analyzing solvency metrics and balance sheet strength using quadrant analysis and Bayesian estimation.


In [120]:
leverage_results = bayesian_category_analysis(df, 'Leverage & Liquidity', FEATURE_CATEGORIES['Leverage & Liquidity'])
leverage_distributions = fit_distributions_by_category(df, 'Leverage & Liquidity',
                                                       FEATURE_CATEGORIES['Leverage & Liquidity'])
create_leverage_liquidity_quadrant(df).show()


### 3.6 Analyst Sentiment
Analyzing analyst recommendations, price target upside, and EPS revision momentum.


In [76]:
sentiment_results = bayesian_category_analysis(df, 'Analyst Sentiment', FEATURE_CATEGORIES['Analyst Sentiment'])
sentiment_distributions = fit_distributions_by_category(df, 'Analyst Sentiment',
                                                        FEATURE_CATEGORIES['Analyst Sentiment'])
create_analyst_sentiment_histogram(df).show()

In [122]:
create_analyst_upside_scatter(df).show()


### 3.7 Earnings Quality
Evaluating earnings surprises, GAAP adjustments, and earnings trajectory with Bayesian methods.


In [78]:
earnings_quality_results = bayesian_category_analysis(df, 'Earnings Quality', FEATURE_CATEGORIES['Earnings Quality'])
earnings_quality_distributions = fit_distributions_by_category(df, 'Earnings Quality',
                                                               FEATURE_CATEGORIES['Earnings Quality'])
earnings_beat_probs = bayesian_earnings_beat_model(df)
create_eps_surprise_histogram(df).show()

In [123]:
create_eps_trajectory_scatter(df).show()


### 3.8 Growth Metrics
Analyzing revenue, EBITDA, EPS, and FCF growth patterns with distribution fitting.


In [80]:
growth_results = bayesian_category_analysis(df, 'Growth Metrics', FEATURE_CATEGORIES['Growth Metrics'])
growth_distributions = fit_distributions_by_category(df, 'Growth Metrics', FEATURE_CATEGORIES['Growth Metrics'])
create_growth_correlation_heatmap(df, FEATURE_CATEGORIES['Growth Metrics']).show()

In [124]:
create_revenue_vs_eps_growth_scatter(df).show()


### 3.9 Cash Flow
Analyzing free cash flow metrics, self-funding ratios, and cash flow quality.


In [82]:
cashflow_results = bayesian_category_analysis(df, 'Cash Flow', FEATURE_CATEGORIES['Cash Flow'])
cashflow_distributions = fit_distributions_by_category(df, 'Cash Flow', FEATURE_CATEGORIES['Cash Flow'])
create_fcf_trajectory_chart(df).show()

In [125]:
create_fcf_margin_yield_scatter(df).show()

In [84]:
create_cash_flow_quality_boxplot(df).show()


### 3.10 Dividend Features
Evaluating dividend sustainability, payout ratios, and shareholder yield.


In [85]:
dividend_results = bayesian_category_analysis(df, 'Dividend Features', FEATURE_CATEGORIES['Dividend Features'])
dividend_distributions = fit_distributions_by_category(df, 'Dividend Features', FEATURE_CATEGORIES['Dividend Features'])
create_dividend_streak_timeline(df).show()

In [126]:
create_dividend_yield_payout_scatter(df).show()

In [87]:
create_shareholder_yield_histogram(df).show()


### 3.11 R&D Investment
Analyzing R&D intensity, growth patterns, and innovation investment efficiency.


In [88]:
rnd_results = bayesian_category_analysis(df, 'R&D Investment', FEATURE_CATEGORIES['R&D Investment'])
rnd_distributions = fit_distributions_by_category(df, 'R&D Investment', FEATURE_CATEGORIES['R&D Investment'])
create_rnd_intensity_boxplot(df).show()

In [127]:
create_rnd_intensity_growth_scatter(df).show()

In [90]:
create_rnd_per_employee_histogram(df).show()


### 3.12 Inventory Temporal
Analyzing inventory cycles, turnover efficiency, and buildup patterns.


In [91]:
inventory_results = bayesian_category_analysis(df, 'Inventory Temporal', FEATURE_CATEGORIES['Inventory Temporal'])
inventory_distributions = fit_distributions_by_category(df, 'Inventory Temporal',
                                                        FEATURE_CATEGORIES['Inventory Temporal'])
create_inventory_cycle_analysis(df).show()

In [128]:
create_inventory_days_turnover_scatter(df).show()


### 3.13 Goodwill & M&A
Evaluating acquisition activity, goodwill concentration, and impairment risk.


In [93]:
goodwill_results = bayesian_category_analysis(df, 'Goodwill & M&A', FEATURE_CATEGORIES['Goodwill & M&A'])
goodwill_distributions = fit_distributions_by_category(df, 'Goodwill & M&A', FEATURE_CATEGORIES['Goodwill & M&A'])
create_goodwill_concentration_boxplot(df).show()

In [129]:
create_goodwill_impairment_scatter(df).show()

In [95]:
create_acquisition_activity_histogram(df).show()


### 3.14 CapEx & Investment
Analyzing capital expenditure patterns, investment efficiency, and M&A intensity.


In [130]:
capex_results = bayesian_category_analysis(df, 'CapEx & Investment', FEATURE_CATEGORIES['CapEx & Investment'])
capex_distributions = fit_distributions_by_category(df, 'CapEx & Investment', FEATURE_CATEGORIES['CapEx & Investment'])
create_capex_growth_scatter(df).show()

In [97]:
create_investment_efficiency_boxplot(df).show()

In [98]:
create_ma_intensity_histogram(df).show()


### 3.15 Probability Analytics: Earnings Beat & EPS Streaks
Advanced probability analysis using Bayesian Beta-Binomial models for earnings beat prediction,
Markov chain-style EPS streak analysis, and model confidence estimation with calibration metrics.


In [99]:
import numpy as np

# Initialize probability analytics models
beat_model = EarningsBeatProbabilityModel()
streak_analyzer = EPSStreakAnalyzer(mean_reversion_weight=0.3)
confidence_estimator = ModelConfidenceEstimator(n_bins=10)

# Prepare data for analysis - create proxy columns from eps_trajectory_score if needed
df_prob = df.copy()
if 'eps_trajectory_score' in df_prob.columns:
    df_prob['eps_beat_count'] = (df_prob['eps_trajectory_score'].fillna(50) / 100 * 5).astype(int)
    df_prob['eps_total_reports'] = 5

# Compute Bayesian earnings beat probabilities
probability_results = beat_model.analyze_dataframe(
    df_prob,
    beats_col='eps_beat_count',
    total_col='eps_total_reports',
    sector_col='sector' if 'sector' in df_prob.columns else 'industry',
    ticker_col='ticker'
)

print(f"📊 Earnings Beat Probability Analysis")
print(f"   Stocks analyzed: {len(probability_results)}")
if len(probability_results) > 0:
    likely_beat = (probability_results['beat_classification'] == 'likely_beat').sum()
    print(f"   Classified as 'likely beat': {likely_beat} ({likely_beat / len(probability_results) * 100:.1f}%)")
    print(f"   Mean posterior beat probability: {probability_results['posterior_beat_prob'].mean():.1%}")
    print(f"   Mean confidence score: {probability_results['confidence_score'].mean():.2f}")


📊 Earnings Beat Probability Analysis
   Stocks analyzed: 1901
   Classified as 'likely beat': 1275 (67.1%)
   Mean posterior beat probability: 56.9%
   Mean confidence score: 0.36


In [100]:
# Display top stocks by posterior beat probability
if len(probability_results) > 0:
    top_beat_prob = probability_results.nlargest(15, 'posterior_beat_prob')[
        ['ticker', 'sector', 'historical_beat_rate', 'posterior_beat_prob',
         'ci_90_lower', 'ci_90_upper', 'confidence_score', 'beat_classification']
    ]
    display(top_beat_prob)


,ticker,sector,historical_beat_rate,posterior_beat_prob,ci_90_lower,ci_90_upper,confidence_score,beat_classification
196,LAGRB,Information Technology,1.0,0.850000,0.639330,0.980081,0.400,likely_beat
356,VAIAS,Information Technology,1.0,0.850000,0.639330,0.980081,0.400,likely_beat
537,CER,Information Technology,1.0,0.850000,0.639330,0.980081,0.400,likely_beat
1203,REY,Information Technology,1.0,0.850000,0.639330,0.980081,0.400,likely_beat
1331,ACN,Information Technology,1.0,0.850000,0.639330,0.980081,0.400,likely_beat
1368,SCT,Information Technology,1.0,0.850000,0.639330,0.980081,0.400,likely_beat
77,NOVOB,Health Care,1.0,0.842105,0.622166,0.978885,0.375,likely_beat
173,MCAP,Health Care,1.0,0.842105,0.622166,0.978885,0.375,likely_beat
279,1SXP,Health Care,1.0,0.842105,0.622166,0.978885,0.375,likely_beat
612,ALKB,Health Care,1.0,0.842105,0.622166,0.978885,0.375,likely_beat


In [101]:
# Earnings Beat Probability Dashboard
create_earnings_probability_dashboard(probability_results).show()


#### EPS Streak Analysis
Analyzing earnings beat/miss streaks with continuation and mean reversion probabilities.


In [102]:
# Compute EPS streak analysis
streak_results = streak_analyzer.analyze_dataframe(
    df,
    trajectory_col='eps_trajectory_score',
    streak_col='eps_positive_streak' if 'eps_positive_streak' in df.columns else None,
    ticker_col='ticker'
)

print(f"📈 EPS Streak Analysis")
print(f"   Stocks analyzed: {len(streak_results)}")
if len(streak_results) > 0:
    beat_streaks = (streak_results['streak_type'] == 'beat').sum()
    miss_streaks = (streak_results['streak_type'] == 'miss').sum()
    print(f"   On beat streaks: {beat_streaks}")
    print(f"   On miss streaks: {miss_streaks}")
    print(f"   Mean continuation probability: {streak_results['continuation_probability'].mean():.1%}")
    print(f"   Mean reversion probability: {streak_results['mean_reversion_probability'].mean():.1%}")


📈 EPS Streak Analysis
   Stocks analyzed: 1901
   On beat streaks: 1671
   On miss streaks: 230
   Mean continuation probability: 38.8%
   Mean reversion probability: 61.2%


In [103]:
# Display stocks with strongest beat streaks
if len(streak_results) > 0:
    strong_streaks = streak_results[streak_results['streak_type'] == 'beat'].nlargest(15, 'current_streak')[
        ['ticker', 'current_streak', 'streak_type', 'continuation_probability',
         'mean_reversion_probability', 'expected_next_outcome', 'prediction_confidence']
    ]
    display(strong_streaks)


,ticker,current_streak,streak_type,continuation_probability,mean_reversion_probability,expected_next_outcome,prediction_confidence
0,ELISA,5,beat,0.288408,0.711592,miss,0.5
1,ALV,5,beat,0.288408,0.711592,miss,0.5
2,LIFCOB,5,beat,0.288408,0.711592,miss,0.5
3,SKFB,5,beat,0.288408,0.711592,miss,0.5
4,COLR,5,beat,0.288408,0.711592,miss,0.5
5,ARJOB,5,beat,0.288408,0.711592,miss,0.5
7,LIGHT,5,beat,0.288408,0.711592,miss,0.5
8,ABDP,5,beat,0.288408,0.711592,miss,0.5
10,CEK,5,beat,0.288408,0.711592,miss,0.5
12,ALTPC,5,beat,0.288408,0.711592,miss,0.5


In [104]:
# EPS Streak Analysis Chart
create_eps_streak_analysis_chart(streak_results).show()


#### Model Confidence & Calibration
Assessing model reliability using Brier score, calibration error, and AUC-ROC metrics.


In [105]:
# Compute model confidence metrics (using simulated outcomes for demonstration)
if len(probability_results) > 10:
    np.random.seed(42)
    # Simulate actual outcomes based on posterior probability
    simulated_outcomes = (
            np.random.random(len(probability_results))
            < probability_results['posterior_beat_prob'].values
    ).astype(float)

    confidence_result = confidence_estimator.compute_confidence_metrics(
        predicted_probs=probability_results['posterior_beat_prob'].values,
        actual_outcomes=simulated_outcomes,
        model_name='Bayesian Earnings Beat Model'
    )

    print(f"🎯 Model Confidence Metrics")
    print(f"   Brier Score: {confidence_result.brier_score:.4f} (lower is better, 0=perfect)")
    print(f"   Calibration Error (ECE): {confidence_result.calibration_error:.4f}")
    print(f"   Discrimination (AUC-ROC): {confidence_result.discrimination_auc:.3f}")
    print(f"   Overall Confidence: {confidence_result.overall_confidence:.1f}/100")


🎯 Model Confidence Metrics
   Brier Score: 0.2308 (lower is better, 0=perfect)
   Calibration Error (ECE): 0.0230
   Discrimination (AUC-ROC): 0.643
   Overall Confidence: 78.1/100


In [106]:
# Model Confidence Calibration Chart
if 'confidence_result' in dir():
    create_confidence_calibration_chart(confidence_result).show()


## 4. Advanced Modeling: Hierarchical Bayes & Copulas
Modeling sector-level dependencies and tail correlations between Valuation and Quality.


In [107]:
roe_hierarchical = hierarchical_mcmc_by_sector(df, 'roe', sector_col='industry')
copula_fit = fit_gaussian_copula(df, ['p_e_ratio', 'piotroski_f_score'])


## 5. Earnings Calendar Analysis
Visualizing upcoming earnings dates with quality overlay.


In [108]:
create_earnings_calendar_heatmap(df).show()


## 6. Enhanced Visualizations
New advanced visualizations for comprehensive analysis.


In [109]:
# Valuation violin plot by industry
create_valuation_violin_plot(df).show()


In [131]:
# Leverage vs liquidity bubble chart
create_leverage_liquidity_bubble_chart(df).show()


In [111]:
# Quality radar chart for top stock
if len(df) > 0:
    top_ticker = df.iloc[0]['ticker']
    print(f"Quality Radar for: {top_ticker}")
    create_quality_risk_radar_chart(df, top_ticker).show()


## 7. Stock Screening & Summary
Final ranking and interactive dashboard for the top opportunities using multiple screening strategies.


### 7.1 Enhanced Quality Screener


In [112]:
# Quality screening with multiple criteria
quality_stocks = create_enhanced_screener(df, min_fscore=7, min_fcf_positive_years=4)
print(f"🏆 Quality Screen: {len(quality_stocks)} stocks found")
if len(quality_stocks) > 0:
    display(quality_stocks[['ticker', 'name', 'industry', 'piotroski_f_score',
                            'distress_risk_score', 'fcf_positive_years']].head(15))


,ticker,name,industry,piotroski_f_score,distress_risk_score,fcf_positive_years
405,VERK,Verkkokauppa.com Oyj,Broadline Retail,9,100.000000,4
376,RAYB,RaySearch Laboratories AB (publ),Health Care Technology,9,100.000000,5
307,YPSN,Ypsomed Holding AG,Health Care Equipment and Supplies,9,100.000000,4
1596,ACP,Asseco Poland S.A.,Software,9,38.333333,5
1597,MBB,MBB SE,Industrial Conglomerates,9,100.000000,4
711,MTELEKOM,Magyar Telekom Távközlési Nyilvánosan Müködö R...,Diversified Telecommunication Services,9,100.000000,5
731,ACS,ACS Actividades de Construcción y Servicios S.A.,Construction and Engineering,9,100.000000,4
578,JMAT,Johnson Matthey Plc,Chemicals,9,100.000000,5
1749,TEL,TE Connectivity plc,Electronic Equipment Instruments and Components,9,100.000000,5
1309,SHO,Shoper S.A.,Software,9,100.000000,5


### 7.2 GARP (Growth at Reasonable Price) Screener


In [113]:
# GARP opportunities
garp_stocks = screen_garp_opportunities(df)
print(f"📈 GARP Screen: {len(garp_stocks)} stocks found")
if len(garp_stocks) > 0:
    cols = ['ticker', 'name', 'industry']
    if 'peg_ratio' in garp_stocks.columns:
        cols.append('peg_ratio')
    if 'eps_yoy_growth' in garp_stocks.columns:
        cols.append('eps_yoy_growth')
    if 'p_e_ratio' in garp_stocks.columns:
        cols.append('p_e_ratio')
    display(garp_stocks[cols].head(15))


,ticker,name,industry,peg_ratio,eps_yoy_growth,p_e_ratio
1691,RBW,Rainbow Tours S.A.,Hotels Restaurants and Leisure,0.059027,55.298013,9.2
1633,ALFLE,Fleury Michon SA,Food Products,0.073196,336.029412,8.9
1367,PDD,PDD Holdings Inc.,Broadline Retail,0.077737,78.080000,9.7
1860,FLU,Flughafen Wien Aktiengesellschaft,Transportation Infrastructure,0.088585,20.270270,24.5
934,SGF,Sogefi S.p.A.,Automobile Components,0.096362,127.777778,28.4
1210,RWE,RWE Aktiengesellschaft,Independent Power and Renewable Electricity Pr...,0.110163,217.777778,8.9
1491,AT,Ashtead Technology Holdings Plc,Trading Companies and Distributors,0.115732,32.352941,12.5
347,IVG,Iveco Group N.V.,Machinery,0.132405,51.685393,11.0
1034,GSL,Global Ship Lease Inc.,Marine Transportation,0.139542,16.926771,3.9
1168,FOUR,4imprint Group plc,Media,0.184306,10.052910,13.5


### 7.3 High-Yield Safe Dividend Screener


In [114]:
# Safe high-yield dividends
safe_div_stocks = screen_high_yield_safe_dividends(df)
print(f"💰 Safe High-Yield Dividend Screen: {len(safe_div_stocks)} stocks found")
if len(safe_div_stocks) > 0:
    yield_col = 'dividend_yield_ltm' if 'dividend_yield_ltm' in safe_div_stocks.columns else 'dividend_yield'
    cols = ['ticker', 'name', 'industry', yield_col]
    if 'dividend_payout_ratio' in safe_div_stocks.columns:
        cols.append('dividend_payout_ratio')
    if 'distress_risk_score' in safe_div_stocks.columns:
        cols.append('distress_risk_score')
    display(safe_div_stocks[cols].head(15))


💰 Safe High-Yield Dividend Screen: 0 stocks found


### 7.4 Value Opportunities Screener


In [132]:
# Value opportunities
value_stocks = screen_value_opportunities(df, max_pe_ratio=20, min_upside_potential=15)
print(f"💎 Value Screen: {len(value_stocks)} stocks found")
if len(value_stocks) > 0:
    cols = ['ticker', 'name', 'industry']
    if 'p_e_ratio' in value_stocks.columns:
        cols.append('p_e_ratio')
    if 'upside_potential' in value_stocks.columns:
        cols.append('upside_potential')
    if 'fcf_yield' in value_stocks.columns:
        cols.append('fcf_yield')
    display(value_stocks[cols].head(15))


,ticker,name,industry,p_e_ratio,upside_potential,fcf_yield
1452,PHAR,Pharos Energy plc,Oil Gas and Consumable Fuels,5.5,126.055046,19.684083
1697,FUM,Franchi Umberto Marmi S.p.A.,Construction Materials,13.8,119.321149,1.600054
1705,RE,R.E.A. Holdings plc,Food Products,4.5,99.143911,1.643567
506,G5EN,G5 Entertainment AB (publ),Entertainment,7.7,94.355698,24.508376
112,PRICB,Pricer AB (publ),Electronic Equipment Instruments and Components,6.3,88.737624,9.514278
895,NEAG,naturenergie holding AG,Electric Utilities,7.6,83.486239,9.592578
285,EOLUB,Eolus Aktiebolag (publ),Construction and Engineering,6.8,73.333333,62.858650
1891,MIKN,Mikron Holding AG,Machinery,12.1,68.981481,8.619859
1768,MAK,Makarony Polskie S.A.,Food Products,9.1,67.714885,4.183701
567,SUS,Surgical Science Sweden AB (publ),Health Care Equipment and Supplies,13.1,66.543301,3.942376


### 7.5 Growth Momentum Screener


In [116]:
# Growth momentum stocks
growth_stocks = screen_growth_momentum(df, min_revenue_growth=5)
print(f"🚀 Growth Momentum Screen: {len(growth_stocks)} stocks found")
if len(growth_stocks) > 0:
    cols = ['ticker', 'name', 'industry']
    if 'revenue_growth_yoy' in growth_stocks.columns:
        cols.append('revenue_growth_yoy')
    if 'eps_yoy_growth' in growth_stocks.columns:
        cols.append('eps_yoy_growth')
    if 'long_term_trend_score' in growth_stocks.columns:
        cols.append('long_term_trend_score')
    display(growth_stocks[cols].head(15))


🚀 Growth Momentum Screen: 0 stocks found


### 7.6 Summary Dashboard


In [117]:
create_summary_dashboard(df).show()


### 7.7 Export Probability Analytics Results
Export earnings beat probability analysis, EPS streak analysis, and model confidence metrics to CSV files.


In [118]:
from pathlib import Path

output_dir = Path('outputs/analytics')
output_dir.mkdir(parents=True, exist_ok=True)

# Export probability analytics results
if len(probability_results) > 0 and len(streak_results) > 0:
    export_paths = export_probability_analytics_results(
        probability_df=probability_results,
        streak_df=streak_results,
        output_dir=output_dir,
        confidence_result=confidence_result if 'confidence_result' in dir() else None
    )
    print("📁 Exported Probability Analytics Results:")
    for name, path in export_paths.items():
        print(f"   ✓ {name}: {path}")


📁 Exported Probability Analytics Results:
   ✓ probability_analysis: outputs\analytics\earnings_beat_probability_analysis.csv
   ✓ streak_analysis: outputs\analytics\eps_streak_analysis.csv
   ✓ confidence_metrics: outputs\analytics\model_confidence_metrics.csv
   ✓ summary: outputs\analytics\probability_analytics_summary.csv


## 8. Analysis Summary


In [119]:
# Optimization Summary
opt_status = get_optimization_status()
print("=" * 60)
print("📊 ANALYSIS COMPLETE")
print("=" * 60)
print(f"\n🔧 Environment:")
print(f"   JIT Acceleration: {opt_status.get('numba_available')}")
print(f"   Feature Categories: {len(FEATURE_CATEGORIES)}")
print(f"   Total Stocks Analyzed: {len(df)}")

print(f"\n📈 Probability Analytics Summary:")
if len(probability_results) > 0:
    likely_beat = (probability_results['beat_classification'] == 'likely_beat').sum()
    print(f"   Earnings Beat Analysis: {len(probability_results)} stocks")
    print(f"   Likely Beat Classification: {likely_beat} stocks ({likely_beat / len(probability_results) * 100:.1f}%)")
    print(f"   Mean Posterior Beat Prob: {probability_results['posterior_beat_prob'].mean():.1%}")
if len(streak_results) > 0:
    print(f"   EPS Streak Analysis: {len(streak_results)} stocks")
    print(f"   Beat Streaks: {(streak_results['streak_type'] == 'beat').sum()}")
    print(f"   Miss Streaks: {(streak_results['streak_type'] == 'miss').sum()}")
if 'confidence_result' in dir():
    print(f"   Model Confidence Score: {confidence_result.overall_confidence:.1f}/100")

print(f"\n📁 Output Directory: outputs/analytics/")


📊 ANALYSIS COMPLETE

🔧 Environment:
   JIT Acceleration: False
   Feature Categories: 14
   Total Stocks Analyzed: 1901

📈 Probability Analytics Summary:
   Earnings Beat Analysis: 1901 stocks
   Likely Beat Classification: 1275 stocks (67.1%)
   Mean Posterior Beat Prob: 56.9%
   EPS Streak Analysis: 1901 stocks
   Beat Streaks: 1671
   Miss Streaks: 230
   Model Confidence Score: 78.1/100

📁 Output Directory: outputs/analytics/
